In [1]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.flights.flight_type import get_flights

target_date = date(2026,3,1)
icao = "a01180"
df_flights =  get_flights(target_date, algorithm="opensky")
df_adsb = read_adsb(target_date, source="opensky", icaos=[icao])

In [2]:
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from flights.evaluation.visual import visualize_opensky_failure
df_f = df_flights.filter(pl.col("icao") == icao)
df_a = df_adsb.filter(pl.col("icao") == icao)
fig_widget = go.FigureWidget(visualize_opensky_failure(df_f, df_a))

# Plotly sends an internal map._derived value after panning; it is not part
# of the Python layout schema, so discard it while retaining center and zoom.
fig_widget.unobserve_all(name="_js2py_relayout")
def sync_map_view(change):
    message = change["new"]
    if not message:
        fig_widget._js2py_relayout = None
        return
    relayout_data = {
        key: value
        for key, value in message["relayout_data"].items()
        if key not in {"lastInputTime", "map._derived"}
    }
    if relayout_data:
        fig_widget.plotly_relayout(relayout_data, source_view_id=message["source_view_id"])
    fig_widget._js2py_relayout = None
fig_widget.observe(sync_map_view, names="_js2py_relayout")

export_button = widgets.Button(description="Export current map view to PDF", button_style="primary")
export_status = widgets.HTML("<i>Pan or zoom the map, then click Export.</i>")

def export_current_map_view(_):
    pdf_path = Path(f"tmp/opensky_flight_assignment_failure_{icao}.pdf")
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    export_fig = go.Figure(fig_widget)
    export_fig.write_image(pdf_path, format="pdf")
    export_status.value = f"<b>Exported:</b> {pdf_path}"

export_button.on_click(export_current_map_view)
display(widgets.VBox([fig_widget, widgets.HBox([export_button, export_status])]))

    'data': [{'hovertemplate': '%{x|%H:%M:%S}<br>%{y:,.0f} ft<extra>F1: Flight 1…